# writeFeatures — feature-in-node rank tester

Given a **people subset** and a **target feature** `(layer, fidx)`, measure where that
feature ranks among the **direct input features** of an output-token node (e.g. the
` August` logit), across all 46 bio templates and all people. Reports a rank histogram,
a strict "meaningful across tokens" metric, and the common co-influencer features.

**To use:** run the loading cells once, then edit only the **MODEL CONFIG** cell (rarely)
and the **EDIT THIS CELL** cell (people + feature), and run the final cell.

All logic lives in the tested module `clts/writefeatures.py`; this notebook is a thin wrapper.
Kernel: the `clts/.venv-ct` interpreter.

In [ ]:
# ===================== MODEL CONFIG (edit to swap models) =====================
from pathlib import Path
import sys

REPO = Path.cwd()
if REPO.name == "clts":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

MODEL_DIR = REPO / "model/grid-L4-H6"
CLT_DIR   = REPO / "clts/clt_runs/grid-L4-H6/mult16_l02_lr0.0001_ep50_n10000/final"
DATA_DIR  = REPO / "data/bioS_N-Bd_final_grid"
SCAN_NAME = "grid-L4-H6"   # also namespaces the cache/report dir
DEVICE    = "cpu"
# ==============================================================================
print("model config set:", SCAN_NAME)

In [ ]:
import importlib
import torch

import clts.writefeatures as wf
importlib.reload(wf)   # pick up edits to writefeatures.py without a kernel restart

from clts.export_tokenizer import ensure_hf_tokenizer
from clts.load_replacement_model import load_replacement_model
from clts.storage import storage_root
from util.bio_sampler import BioSampler
from util.condensed_tokenizer import CondensedTokenizer

ct = CondensedTokenizer.from_remap_path(DATA_DIR / "old_to_new.json")
sampler = BioSampler(DATA_DIR / "people.json", fields=("birthday",))

if globals().get("model") is None:
    model = load_replacement_model(
        MODEL_DIR, CLT_DIR, ensure_hf_tokenizer(DATA_DIR), SCAN_NAME, device=DEVICE)
    print(f"model loaded — n_layers={model.cfg.n_layers}, d_vocab={model.cfg.d_vocab}")
else:
    print("model already loaded (set `model = None` and re-run to reload)")

CACHE_DIR = storage_root() / "clt_feature_explorer" / SCAN_NAME / "hyptest"

# Tiny partials so the EDIT cell reads cleanly.
people_in_month = lambda month: wf.people_in_month(sampler, ct, month)
people_by_ids   = lambda ids:   wf.people_by_ids(sampler, ids)
people_by_idx   = lambda idxs:  wf.people_by_idx(sampler, idxs)
sample_in_month = lambda month, n, seed=0: wf.sample_in_month(sampler, ct, month, n, seed)
print("helpers ready · cache dir:", CACHE_DIR)

In [ ]:
# ===================== EDIT THIS CELL =====================
PEOPLE          = people_in_month("August")  # or people_by_ids([...]) / people_by_idx([...]) / sample_in_month("August", 20)
TARGET_FEATURE  = (3, 4768)                  # (layer, feature_idx) to locate
TARGET          = "month"                    # "month" = each person's own birth-month token; or pin e.g. " August"
TEMPLATES       = "all"                      # "all" 46 templates, or [0, 5, 12], or ["{name} popped out on {birthday}."]
N_PEOPLE_CAP    = 20                         # cap; capping RANDOM-SAMPLES the pool (None = use all)
SEED            = 0                           # RNG seed for the random people sample
TOP_K           = 10                         # co-influencer table depth (does NOT change the rank histogram)
MULTI_TOK_TOP_K = 5                           # node-level top-K for the "meaningful across tokens" metric (strict)
POS_SPAN_FLAG   = 3                          # loose flag: feature fired at >=2 positions spanning >= this many tokens
RANK_BY_ABS     = False                      # False = signed (promoters first); True = |edge|
SUBSET_LABEL    = "august"                   # short label for the saved report filename
TEMPLATE_WORD_LABELS = {"born", "birth", "day", "date"}  # template words kept literal; others -> template:other
INCLUDE_TOKEN_NODES  = True                              # include tok@* rows in the unified view (expected near-zero)
# ==========================================================
print(f"{len(PEOPLE)} people in pool · target feature {TARGET_FEATURE} · target {TARGET!r}")

In [ ]:
result = wf.run_hypothesis(
    model, sampler, ct, PEOPLE, target_feature=TARGET_FEATURE, target=TARGET,
    templates=TEMPLATES, cache_dir=CACHE_DIR, n_cap=N_PEOPLE_CAP, seed=SEED,
    top_k=TOP_K, multi_tok_top_k=MULTI_TOK_TOP_K, pos_span_flag=POS_SPAN_FLAG,
    rank_by_abs=RANK_BY_ABS, template_word_labels=TEMPLATE_WORD_LABELS,
    include_token_nodes=INCLUDE_TOKEN_NODES,
)

config = {"target_feature": list(TARGET_FEATURE), "target": TARGET,
          "templates": TEMPLATES, "n_cap": N_PEOPLE_CAP, "seed": SEED,
          "top_k": TOP_K, "multi_tok_top_k": MULTI_TOK_TOP_K,
          "pos_span_flag": POS_SPAN_FLAG, "rank_by_abs": RANK_BY_ABS,
          "template_word_labels": sorted(TEMPLATE_WORD_LABELS),
          "include_token_nodes": INCLUDE_TOKEN_NODES,
          "subset_label": SUBSET_LABEL, "scan": SCAN_NAME}
report = wf.build_report(result, top_k=TOP_K, pos_span_flag=POS_SPAN_FLAG,
                         multi_tok_top_k=MULTI_TOK_TOP_K, config=config)
print(wf.format_report(report))

slug = f"{SUBSET_LABEL}-n{N_PEOPLE_CAP}-s{SEED}"
paths = wf.save_report(report, result["records"], CACHE_DIR,
                       layer=TARGET_FEATURE[0], fidx=TARGET_FEATURE[1], subset_slug=slug)
print("\nsaved:", paths["json"])
print("saved:", paths["csv"])